# Exercises with regularisation

## Exercise 1: L2 regularisation and bias-variance tradeoff
This exercise explores the effect of L2 regularisation on fitting/overfitting and on the bias-variance tradeoff.

We will use a simple 2d dataset and a simple polynomial model. The dataset is generated from a known function with some added noise.

In [1]:
# make a simple 2D dataset
import numpy as np
import pandas as pd
np.random.seed(42)

X = np.linspace(1, 3, 10)
y_obs = np.sin(3*X) + 0.5*X**2 + np.random.normal(0, 0.4, size=X.shape)
y_true = lambda x: np.sin(3*x) + 0.5*x**2

df = pd.DataFrame({'x': X-2, 'y': y_obs})

In [2]:
# a simple polynomial regression model
import torch
import torch.nn as nn

class PolynomialModel(nn.Module):
    def __init__(self, degree):
        super().__init__()
        self.poly = nn.Linear(degree, 1)
        self.degree = degree

    def forward(self, x):
        # create polynomial features
        x_poly = torch.cat([x**i for i in range(1, self.degree + 1)], dim=1)    # each feature is a power of x and has a weight
        return self.poly(x_poly)

In [3]:
# Function to calculate bias
def bias(y_true, y_pred):
    return torch.mean(y_pred - y_true)

# Function to compute variance
def variance(y_true, y_pred):
    return torch.var(y_pred - y_true)

# Function to compute MSE (not the loss but the actual MSE)
def mse(bias, variance):
    return bias ** 2 + variance

In [ ]:
# plotting function with animation
import matplotlib.pyplot as plt
from matplotlib import gridspec, animation
from IPython.display import HTML, display

def plot_fit_status(data, x_lin, loss_hist, bias_hist, var_hist, mse_hist, y_model_history):
    """Animated visualization of training progress"""
    fig = plt.figure(figsize=(10, 5))
    gs = gridspec.GridSpec(2, 3)
    ax_main, ax_loss, ax_bivar = fig.add_subplot(gs[:, :-1]), fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[1, 2])
    
    # Convert to numpy arrays for slicing
    loss_hist = np.array(loss_hist)
    bias_hist = np.array(bias_hist)
    var_hist = np.array(var_hist)
    mse_hist = np.array(mse_hist)
    
    # main plot setup
    ax_main.scatter(data['x'], data['value'], alpha=0.6)
    ax_main.plot(x_lin, y_true(x_lin+2), color='grey', linestyle='dashed', alpha=0.3, label='True')
    line_model, = ax_main.plot([], [], color='blue', label='Model')
    ax_main.set_xlabel('x')
    ax_main.set_ylabel('y')
    ax_main.set_ylim(-0.1, 5.1)
    ax_main.grid(alpha=0.3)
    ax_main.legend()
    
    # loss plot setup
    line_loss, = ax_loss.plot([], [], label='Loss', color='C0')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Loss')
    ax_loss.set_xlim(0, len(y_model_history) - 1)
    ax_loss.set_ylim(0, max(loss_hist) * 1.1)
    ax_loss.grid(alpha=0.3)
    ax_loss.legend()
    
    # bias-variance plot setup
    line_bias, = ax_bivar.plot([], [], label='Bias', color='C0')
    line_var, = ax_bivar.plot([], [], label='Variance', color='C1')
    line_mse, = ax_bivar.plot([], [], label='MSE', color='C2')
    ax_bivar.set_xlabel('Epoch')
    ax_bivar.set_ylabel('Value')
    ax_bivar.set_xlim(0, len(y_model_history) - 1)
    ax_bivar.set_ylim(0, max(max(bias_hist), max(var_hist), max(mse_hist)) * 1.1)
    ax_bivar.legend()
    ax_bivar.grid(alpha=0.3)
    
    def update(frame):
        line_model.set_data(x_lin, y_model_history[frame])
        line_loss.set_data(np.arange(frame + 1), loss_hist[:frame + 1])
        line_bias.set_data(np.arange(frame + 1), bias_hist[:frame + 1])
        line_var.set_data(np.arange(frame + 1), var_hist[:frame + 1])
        line_mse.set_data(np.arange(frame + 1), mse_hist[:frame + 1])
        ax_main.set_title(f'Epoch {frame}')
        return line_model, line_loss, line_bias, line_var, line_mse
    
    anim = animation.FuncAnimation(fig, update, frames=len(y_model_history), interval=100, blit=True)
    display(HTML(anim.to_jshtml()))
    plt.close(fig)

In [ ]:
# Define training loop
from tqdm import tqdm

X_lin = np.linspace(-1, 1, 200)
def train_model(model, X, y, epochs=20, lr=0.01, lambda_l2=0.01):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    
    loss_hist = []
    bias_hist = []
    var_hist = []
    mse_hist = []
    y_model_history = []

    X_np = X.numpy().flatten()
    for epoch in tqdm(range(epochs), desc="Training", leave=False):
        model.train()
        model.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred, y)

        if lambda_l2 > 0:
            ### YOUR CODE HERE: add L2 regularisation to the loss

        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())

        bias_val = bias(torch.tensor(y_true(X_np)), y_pred).item()
        var_val = variance(torch.tensor(y_true(X_np)), y_pred).item()
        mse_val = mse(bias_val, var_val)
        bias_hist.append(bias_val)
        var_hist.append(var_val)
        mse_hist.append(mse_val)
        
        with torch.no_grad():
            y_lin = model(torch.tensor(X_lin).float().view(-1, 1))
            y_lin = y_lin.numpy().flatten()
            y_model_history.append(y_lin.copy())
    
    # Plot animation
    plot_fit_status(pd.DataFrame({'x': X_np, 'value': y.numpy().flatten()}), X_lin, 
                    loss_hist, bias_hist, var_hist, mse_hist, y_model_history)
    
    return np.array(loss_hist), np.array(bias_hist), np.array(var_hist), np.array(mse_hist)

# EXPLORE WHAT HAPPENS WHEN YOU CHANGE THESE PARAMETERS
poly_degree = 2
learning_rate = 0.05
lambda_l2 = 0.01

num_epochs = 100

model = PolynomialModel(poly_degree)

X_tensor = torch.tensor(df['x']).float().view(-1, 1)
y_tensor = torch.tensor(df['y']).float().view(-1, 1)

loss_hist, bias_hist, var_hist, mse_hist = train_model(model, X_tensor, y_tensor, epochs=num_epochs, lr=learning_rate, lambda_l2=lambda_l2)

## Exercise 2: Loss and Early Stopping
This exercise is about following the training and validation loss during training, and using early stopping to avoid overfitting.
We will use the generic MNIST dataset and a simple feedforward neural network.

N.B. Training for a meaningfull amount of time on this dataset may take a while depending on your hardware. Also drain power.

In [ ]:
#import the MNIST dataset
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.MNIST(
    root="data",    # Change this directory if needed
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    root="data",    # Change this directory if needed
    train=False,
    download=True,
    transform=ToTensor()
)

print(f"Training data size: {training_data.data.size()}")
print(f"Test data size: {test_data.data.size()}")

In [ ]:
# Simple model
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 10),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
# Training function
import torch
import numpy as np
from tqdm import tqdm

def l2_regularization(model, lambda_l2):
    # YOUR CODE HERE: compute the L2 regularization term for the model
    
    return

def train_model(model, X, y, X_val, y_val, epochs=20, batches=32, lr=0.01, lambda_l2=0.01):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    train_loss_hist = []
    val_loss_hist = []

    for epoch in range(epochs):
        perm = torch.randperm(X.size(0))
        X = X[perm]
        y = y[perm]

        for i in tqdm(range(0, len(X), batches), desc=f"Epoch {epoch}", leave=False):
            X_batch = X[i:i+batches]
            y_batch = y[i:i+batches]
            model.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            if lambda_l2 > 0:
                loss += l2_regularization(model, lambda_l2)

            loss.backward()
            optimizer.step()
        
        with torch.no_grad():
            # YOUR CODE HERE: compute and log train_loss and val_loss


            if epoch % 10 == 0 or epoch == epochs - 1:
                print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
                # torch.save(model.state_dict(), f"model_epoch_{epoch}.pth")
        
        # YOUR CODE HERE: early stopping
        

    return np.array(train_loss_hist), np.array(val_loss_hist)

In [ ]:
model = NeuralNetwork()

X = training_data.data.float()
y = training_data.targets
X_val = test_data.data.float()
y_val = test_data.targets

# PARAMETERS TO EXPLORE
num_epochs = 100
num_batches = 32
learning_rate = 0.005
lambda_l2 = 0.01

train_loss_history, val_loss_history = train_model(model, X, y, X_val, y_val, epochs=num_epochs, batches=num_batches, lr=learning_rate, lambda_l2=lambda_l2)

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(6, 4))
plt.plot(train_loss_history, label='Train Loss')
plt.plot(val_loss_history, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.legend()
plt.show()

In [ ]:
# Calculate accuracy
with torch.no_grad():
    train_pred = model(X)
    train_acc = (train_pred.argmax(dim=1) == y).float().mean().item()
    val_pred = model(X_val)
    val_acc = (val_pred.argmax(dim=1) == y_val).float().mean().item()
print(f"Training Accuracy: {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")

## Exercise 5 (Challenge problem): LASSO algorithm
Rather than the soft order constraint on the squares of the weights, one could use absolute values of the weights:
$$
\min_{w\in \mathbb{R}^d} E_{in}(w) \quad \mathrm{subject\ to\ } \sum^d_{i=0}|w_i|\leq t
$$
Where $t$ is a prespecified free parameter that determines the degree of regularization.

(a) Rewrite the above equation, with square error loss into Lagrangian form. [Hint: see [Khan Academy](https://www.khanacademy.org/math/multivariable-calculus/applications-of-multivariable-derivatives/constrained-optimization/a/lagrange-multipliers-single-constraint) on constrained optimisation with Lagrangian multipliers.]

(b) Discuss why the Lagrangian formulation cannot be solved the usual way (by derivatives and solving for stationary points).

(c) In the code provided below, complete the training loop by adding L1 regularisation.

(d) Explain the dummy data case, experiment with different values of $\lambda$ and discuss what happens to the weights of the model.

In [ ]:
# Generate dummy dataset
import torch
torch.manual_seed(42)

n_samples, n_features = 100, 20
X = torch.randn(n_samples, n_features)

true_weights = torch.tensor([2.0, -1.5, 1.0, -0.8] + [0.0] * (n_features - 4))
y = X @ true_weights + torch.randn(n_samples) * 0.5

In [ ]:
import numpy as np
import torch.nn as nn

class LinearModel(nn.Module):
    def __init__(self, input_dim):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1, bias=False)
    
    def forward(self, x):
        return self.linear(x)

def train_model(model, X, y, lambda_l1, epochs=250, lr=0.01):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()
    weights_history = []
    
    for epoch in range(epochs):
        model.zero_grad()
        y_pred = model(X)
        loss = mse_loss(y_pred, y.view(-1, 1))
        if lambda_l1 > 0:
            # YOUR CODE HERE

        loss.backward()
        optimizer.step()
        weights_history.append(model.linear.weight.detach().numpy().flatten())
    return np.array(weights_history)

In [ ]:
# Exercise setup
model_no_l1 = LinearModel(n_features)
model_l1 = LinearModel(n_features)
lambda_l1 = 0.1

weights_no_l1 = train_model(model_no_l1, X, y, lambda_l1=0.0)
weights_l1 = train_model(model_l1, X, y, lambda_l1=lambda_l1)

# Print final weights
print("Final weights without L1:", weights_no_l1[-1])
print("Final weights with L1:", weights_l1[-1])
print("Number of non-zero weights with L1:", np.sum(np.abs(weights_l1[-1]) > 1e-3))

In [ ]:
# Bit for generating plots of weights
import matplotlib.pyplot as plt

def plot_weights(weights_no_l1, weights_l1, n_features_to_plot=8):
    fig, axs = plt.subplots(1, 2, figsize=(10, 8), sharey=True)
    handles, labels = [], []
    epochs = np.arange(weights_no_l1.shape[0])
    for i in range(n_features_to_plot):
        h = axs[0].plot(epochs, weights_no_l1[:, i], label=f'Feature {i+1}')[0]
        axs[1].plot(epochs, weights_l1[:, i], color=h.get_color())
        handles.append(h)
        labels.append(f'Feature {i+1}')

    axs[0].set_title("Weights without L1 Regularization")
    axs[0].set_xlabel("Epoch")
    axs[0].set_ylabel("Weight Value")

    axs[1].set_title("Weights with L1 Regularization")
    axs[1].set_xlabel("Epoch")

    fig.legend(handles, labels, loc='upper center', ncol=min(n_features_to_plot, 8), bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_weights(weights_no_l1, weights_l1)